# Create Local CA for HTTPS

In [ ]:
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path

from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.x509.oid import NameOID


def create_local_ca(
        output_dir: Path,
        common_name: str = "CLA321 Local CA",
) -> tuple[Path, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=4096,
    )

    subject = issuer = x509.Name(
        [
            x509.NameAttribute(
                NameOID.ORGANIZATION_NAME,
                "CLA321",
            ),
            x509.NameAttribute(
                NameOID.COMMON_NAME,
                common_name,
            ),
        ]
    )

    now = datetime.now(timezone.utc)

    certificate = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(issuer)
        .public_key(private_key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(now - timedelta(minutes=5))
        .not_valid_after(now + timedelta(days=365))
        .add_extension(
            x509.BasicConstraints(
                ca=True,
                path_length=None,
            ),
            critical=True,
        )
        .add_extension(
            x509.KeyUsage(
                digital_signature=True,
                content_commitment=False,
                key_encipherment=False,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=True,
                crl_sign=True,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(
            x509.SubjectKeyIdentifier.from_public_key(
                private_key.public_key()
            ),
            critical=False,
        )
        .add_extension(
            x509.AuthorityKeyIdentifier.from_issuer_public_key(
                private_key.public_key()
            ),
            critical=False,
        )
        .sign(
            private_key=private_key,
            algorithm=hashes.SHA256(),
        )
    )

    key_path = output_dir / "ca.key"
    cert_path = output_dir / "ca.crt"

    key_path.write_bytes(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
    )

    cert_path.write_bytes(
        certificate.public_bytes(
            serialization.Encoding.PEM
        )
    )

    return cert_path, key_path

In [ ]:
cert_path, key_path = create_local_ca(
    Path("./certs/ca")
)

print(f"CA certificate: {cert_path}")
print(f"CA private key: {key_path}")

# Create Server Certificate

In [ ]:
import shutil
from datetime import datetime, timedelta, timezone
from pathlib import Path

from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.x509.oid import NameOID




def create_server_certificate(
        ca_cert_path: Path,
        ca_key_path: Path,
        output_dir: Path,
        hostname: str,
) -> tuple[Path, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    ca_certificate = x509.load_pem_x509_certificate(
        ca_cert_path.read_bytes()
    )

    ca_private_key = serialization.load_pem_private_key(
        ca_key_path.read_bytes(),
        password=None,
    )

    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )

    subject = x509.Name(
        [
            x509.NameAttribute(
                NameOID.ORGANIZATION_NAME,
                "CLA321",
            ),
            x509.NameAttribute(
                NameOID.COMMON_NAME,
                hostname,
            ),
        ]
    )

    now = datetime.now(timezone.utc)

    certificate = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(ca_certificate.subject)
        .public_key(private_key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(now - timedelta(minutes=5))
        .not_valid_after(now + timedelta(days=825))
        .add_extension(
            x509.SubjectAlternativeName(
                [
                    x509.DNSName(hostname),
                ]
            ),
            critical=False,
        )
        .add_extension(
            x509.BasicConstraints(
                ca=False,
                path_length=None,
            ),
            critical=True,
        )
        .add_extension(
            x509.KeyUsage(
                digital_signature=True,
                content_commitment=False,
                key_encipherment=True,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=False,
                crl_sign=False,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(
            x509.SubjectKeyIdentifier.from_public_key(
                private_key.public_key()
            ),
            critical=False,
        )
        .add_extension(
            x509.AuthorityKeyIdentifier.from_issuer_public_key(
                ca_private_key.public_key()
            ),
            critical=False,
        )
        .add_extension(
            x509.ExtendedKeyUsage(
                [
                    x509.oid.ExtendedKeyUsageOID.SERVER_AUTH,
                ]
            ),
            critical=False,
        )
        .sign(
            private_key=ca_private_key,
            algorithm=hashes.SHA256(),
        )
    )

    key_path = output_dir / f"{hostname}.key"
    cert_path = output_dir / f"{hostname}.crt"

    key_path.write_bytes(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
    )

    cert_path.write_bytes(
        certificate.public_bytes(
            serialization.Encoding.PEM
        )
    )

    return cert_path, key_path

In [ ]:
registry_cert, registry_key = create_server_certificate(
    ca_cert_path=Path("./certs/ca/ca.crt"),
    ca_key_path=Path("./certs/ca/ca.key"),
    output_dir=Path("./certs/registry"),
    hostname="registry.gitlab.localhost",
)

gitlab_cert, gitlab_key = create_server_certificate(
    ca_cert_path=Path("./certs/ca/ca.crt"),
    ca_key_path=Path("./certs/ca/ca.key"),
    output_dir=Path("./certs/gitlab"),
    hostname="gitlab.localhost",
)

headlamp_cert, headlamp_key = create_server_certificate(
    ca_cert_path=Path("./certs/ca/ca.crt"),
    ca_key_path=Path("./certs/ca/ca.key"),
    output_dir=Path("./certs/headlamp"),
    hostname="headlamp.localhost",
)

openbao_cert, openbao_key = create_server_certificate(
    ca_cert_path=Path("./certs/ca/ca.crt"),
    ca_key_path=Path("./certs/ca/ca.key"),
    output_dir=Path("./certs/openbao"),
    hostname="openbao.localhost",
)

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path.cwd().parent

SOURCE_DIR = PROJECT_ROOT / "notebooks" / "certs"
TARGET_DIR = PROJECT_ROOT / "traefik" / "certs"

TARGET_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(SOURCE_DIR, TARGET_DIR, dirs_exist_ok=True)

# Upload Local CA Certificate to the K3d

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path.cwd().parent

source = PROJECT_ROOT / "notebooks" / "certs" / "ca" / "ca.crt"
target = PROJECT_ROOT / "k3d" / "certs" / "cla321-local-ca.crt"

target.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, target)

# Trust Local CA

In [ ]:
%%bash

sudo cp notebooks/certs/ca/ca.crt /etc/ca-certificates/trust-source/anchors/cla321-local-ca.crt
sudo update-ca-trust